
# Bikeability Index with OSMnx — Notebook Template

This notebook is a scaffold to help you replicate the workflow we discussed using **OSMnx**, **GeoPandas**, and **Shapely**.

**What you'll do here:**
1) Define your study boundary (place name or polygon)  
2) Download road network from OpenStreetMap with OSMnx  
3) Build your *system of tags* → classify edges for each factor  
4) Create a regular analysis grid (choose your cell size)  
5) Compute factor scores per cell (length-/area-weighted)  
6) Combine with weights to produce your Bikeability Index (BI)  
7) Export results (GeoPackage/GeoJSON) and quick maps



## Environment & dependencies

- Recommended: **conda** environment with Python ≥ 3.10  
- Install packages:
```bash
conda create -n bikeability python=3.11 -y
conda activate bikeability
pip install osmnx geopandas shapely pyproj rtree matplotlib
```
> On Windows/macOS, `pip` wheels should include GEOS/PROJ. If you hit install issues,
> try `conda-forge`: `conda install -c conda-forge osmnx geopandas rtree`.

If you prefer **JupyterLab**:
```bash
pip install jupyterlab
jupyter lab
```
Or classic Notebook:
```bash
pip install notebook
jupyter notebook
```


In [2]:

# Imports
import os
import math
import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
from shapely.geometry import box

ox.settings.use_cache = True
ox.settings.log_console = False

print("Versions -> osmnx:", ox.__version__)
print("geopandas:", gpd.__version__)


Versions -> osmnx: 2.0.3
geopandas: 1.0.1


## 1) Define your study boundary

In [3]:

PLACE = "Munich, Germany"  # <- change me
boundary = ox.geocode_to_gdf(PLACE)
display(boundary)

,geometry,bbox_west,bbox_south,bbox_east,bbox_north,place_id,osm_type,osm_id,lat,lon,class,type,place_rank,importance,addresstype,name,display_name
0,"MULTIPOLYGON (((11.36078 48.15807, 11.36085 48...",11.360777,48.061624,11.72291,48.248116,117173815,relation,62428,48.137108,11.575382,boundary,administrative,12,0.799006,city,Munich,"Munich, Bavaria, Germany"


## 2) Download OSM network and convert to GeoDataFrames

In [4]:
G = ox.graph_from_polygon(boundary.geometry.iloc[0], network_type="all_private", simplify=True)
nodes, edges = ox.graph_to_gdfs(G, nodes=True, edges=True, fill_edge_geometry=True)
print(len(nodes), "nodes;", len(edges), "edges")
display(edges.head(3))


ValueError: Unrecognized network_type 'all_private'.


## 3) Apply your *system of tags* (factor classification)

Below are illustrative selections. Adjust them to mirror your study's tag logic.


In [ ]:

# Exclude service roads (often access/parking)
edges = edges[edges["highway"].astype(str) != "service"]

# Main roads to evaluate separation
MAIN_SET = ["primary","primary_link","secondary","secondary_link"]
main_roads = edges[edges["highway"].isin(MAIN_SET)].copy()

# Cycle facility flag on main roads (lane/track/opposite_* or dedicated cycleway)
cycle_cols = [c for c in edges.columns if c.startswith("cycleway")]
main_roads["cycle_fac"] = (
    main_roads[cycle_cols].notna().any(axis=1)
    | main_roads["highway"].eq("cycleway")
    | (main_roads.get("bicycle_road") == "yes")
)

# Traffic-calmed / bike-friendly set (example — adapt as needed)
CALM_SET = [
    "tertiary","tertiary_link","unclassified","living_street","residential",
    "track","cycleway","path","footway","pedestrian"
]
traffic_calmed = edges[edges["highway"].isin(CALM_SET)].copy()


## 4) Create analysis grid

In [ ]:

# Project to a metric CRS for accurate lengths/areas
boundary_m = boundary.to_crs(3857)
edges_m = edges.to_crs(3857)
main_roads_m = main_roads.to_crs(3857)
traffic_calmed_m = traffic_calmed.to_crs(3857)

# Grid size in meters
CELL = 2000  # 2 km — change as needed

minx, miny, maxx, maxy = boundary_m.total_bounds
polys = []
x = minx
while x < maxx:
    y = miny
    while y < maxy:
        polys.append(box(x, y, x+CELL, y+CELL))
        y += CELL
    x += CELL

grid = gpd.GeoDataFrame(geometry=polys, crs=boundary_m.crs)
grid = gpd.overlay(grid, boundary_m[["geometry"]], how="intersection")
print("Grid cells:", len(grid))
grid.head(2)


## 5) Utility: length-in-cell and scoring helpers

In [ ]:

def clip_and_length(lines_gdf, poly):
    if lines_gdf.empty:
        return 0.0
    clip = gpd.overlay(lines_gdf, gpd.GeoDataFrame(geometry=[poly], crs=lines_gdf.crs), how="intersection")
    return float(clip.length.sum()) if not clip.empty else 0.0

def score_from_bins(value, bins, scores):
    import numpy as np
    # value assumed in percent (0..100) or metric depending on your bins
    idx = np.digitize([value], bins, right=True)[0]
    return scores[idx]


### Example factor: Cycle facility coverage on main roads (R)

In [ ]:

# Example thresholds (edit to match your chosen scale)
R_BINS = [0, 21, 47, 67, 86, 100]  # percentage
R_SCORES = [0, 2, 4, 6, 8, 10]

results = []
for idx, poly in grid.geometry.items():
    total_main = clip_and_length(main_roads_m, poly)
    if total_main == 0:
        r_score = None  # skip later if no main roads in cell
    else:
        main_with = clip_and_length(main_roads_m[main_roads_m["cycle_fac"]], poly)
        share = 100 * main_with / total_main
        r_score = score_from_bins(share, R_BINS, R_SCORES)
    results.append({"cell_id": idx, "R": r_score})

scores_df = pd.DataFrame(results).set_index("cell_id")
grid_scores = grid.join(scores_df)
grid_scores.head(3)



### Add more factors here

Create additional blocks for:
- **V** (traffic-calmed share), 
- **K** (connectivity/intersections), 
- **T** (topography/grade), 
- **O** (surface quality), 
- **G** (green/water share), 
- plus any others you keep (Soft factors).

> Tip: Keep each factor column in `grid_scores` as 0–10 for a consistent weighted sum later.


## 6) Combine factors with weights → BI

In [ ]:

# Example weights (sum to 1.0). Replace with your literature-averaged weights.
WEIGHTS = {
    "V": 0.26,
    "R": 0.21,
    "K": 0.13,
    "T": 0.14,
    "O": 0.13,
    "G": 0.13,
}

# Ensure missing cells (None) don't break the formula by skipping them
for k in WEIGHTS:
    if k not in grid_scores.columns:
        grid_scores[k] = np.nan

# Weighted sum across available factors
def weighted_bi(row, weights):
    vals = []
    wts = []
    for k, w in weights.items():
        v = row[k]
        if pd.notna(v):
            vals.append(v * w)
            wts.append(w)
    return np.nan if not vals else sum(vals) * (1.0 / sum(wts))

grid_scores["BI"] = grid_scores.apply(lambda r: weighted_bi(r, WEIGHTS), axis=1)
grid_scores[["BI"] + list(WEIGHTS.keys())].head(5)


## 7) Export & quick map

In [ ]:

# Export to GeoPackage
out_path = "bikeability.gpkg"
layer_name = "bi_grid"
grid_scores.to_file(out_path, layer=layer_name, driver="GPKG")
print("Saved:", out_path, "layer:", layer_name)

# Quick visualization
ax = grid_scores.plot(column="BI", legend=True, figsize=(8,8), scheme="quantiles", k=5)
boundary_m.boundary.plot(ax=ax, linewidth=1, alpha=0.7)
ax.set_axis_off()



### (Optional) Add elevation/grade with OSMnx

If you want to score **Topography (T)**:  
- Add elevations to edges (requires an elevation service, e.g., ALOS/SRTM tiles or API).  
- Then compute grade per edge and length-weight a score per cell.

See: `ox.add_node_elevations_raster`, `ox.add_edge_grades` in the OSMnx docs.
